# Market data view
Market data of one underlying in the latest `marketdata/` snapshot: summary, forwards and dividends, discount curve,
implied vol surface, correlations.

In [ ]:
TICKER = "TICKER1"        # underlying to show
SNAPSHOT_DATE = None      # None = latest snapshot (or a date, e.g. date(2026, 11, 5))

In [ ]:
from datetime import date, timedelta
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tensorquant as tq

REPO = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "marketdata").is_dir())
mkt = tq.MarketEnvironment.from_folder(REPO / "marketdata", evaluation_date=SNAPSHOT_DATE)
tq.Settings.evaluation_date = mkt.evaluation_date
tickers = sorted(k.split(":")[2] for k in mkt.risk_factors() if k.startswith("EQ:") and k.endswith(":SPOT"))
if TICKER not in tickers:
    raise ValueError(f"{TICKER} not in the snapshot: {tickers}")
eq = mkt.get_equity(TICKER)
S0 = float(eq.spot)
print(f"snapshot {mkt.evaluation_date}, underlyings: {', '.join(tickers)}")

## Summary

In [ ]:
eq.summary()

## Forwards and dividends

In [ ]:
TENORS = {"1M": 1 / 12, "3M": 0.25, "6M": 0.5, "1Y": 1.0, "2Y": 2.0, "3Y": 3.0, "5Y": 5.0, "7Y": 7.0, "10Y": 10.0}
fwd = pd.DataFrame([{"tenor": k, "T": t, "forward": float(eq.forward(t)), "forward / spot": float(eq.forward(t)) / S0,
                     "PV dividends": float(eq.pv_dividends(t)), "implied div yield": float(eq.implied_div_yield(t)),
                     "discount factor": float(eq.discount_curve.discount(t))} for k, t in TENORS.items()])
fwd.set_index("tenor").round(4)

In [ ]:
if eq.dividends is not None:
    divs = pd.DataFrame({"ex date": eq.dividends.ex_dates, "pay date": eq.dividends.payment_dates,
                         "amount": eq.dividends.amounts})
    divs["% of spot"] = 100 * divs["amount"] / S0
    display(divs.round(4))
else:
    print(f"no discrete dividends (dividend model: {eq.dividend_model})")

grid = np.linspace(1 / 365, max(TENORS.values()), 400)
fig, axes = plt.subplots(1, 2, figsize=(13, 3.8))
axes[0].plot(grid, [float(eq.forward(t)) for t in grid])
axes[0].axhline(S0, color="k", ls="--", lw=1, label="spot")
axes[0].set(xlabel="T (years)", ylabel="forward", title=f"{TICKER} forward")
axes[0].legend()
axes[1].plot(grid, [100 * float(eq.implied_div_yield(t)) for t in grid])
axes[1].set(xlabel="T (years)", ylabel="%", title="implied dividend yield")
plt.tight_layout()

## Discount curve

In [ ]:
curve = eq.discount_curve
zeros = pd.DataFrame({"date": curve.dates, "T": curve.pillars, "zero rate (%)": 100 * np.asarray(curve.rates, float)})
print(f"{curve.name} ({eq.currency.name})")
display(zeros.round(4))
t = np.linspace(1 / 365, max(curve.pillars), 400)
plt.figure(figsize=(6.5, 3.5))
plt.plot(t, [100 * float(curve.zero_rate(x)) for x in t], label="zero rate")
plt.plot(curve.pillars, 100 * np.asarray(curve.rates, float), "o", ms=3)
plt.gca().set(xlabel="T (years)", ylabel="%", title=f"{curve.name} zero curve");

## Implied volatility surface

In [ ]:
vs = eq.vol_surface
T = np.asarray(vs.maturity, float)
K = np.asarray(vs.strike, float)
expiries = [mkt.evaluation_date + timedelta(days=int(round(365 * t))) for t in T]
vols = pd.DataFrame(100 * vs.volatility_matrix.numpy(), index=pd.Index(expiries, name="expiry"),
                    columns=pd.Index([f"{100 * k / S0:.0f}%" for k in K], name="strike / spot"))
vols.round(2)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
colors = plt.cm.viridis(np.linspace(0, 1, len(T)))
for i, t in enumerate(T):
    axes[0].plot(100 * K / S0, vols.iloc[i], "o-", ms=3, color=colors[i], label=str(expiries[i]))
axes[0].set(xlabel="strike / spot (%)", ylabel="implied vol (%)", title="smiles")
axes[0].legend(fontsize=7, ncol=2)
atm = [100 * float(vs.volatility(strike=float(eq.forward(t)), tenor=t)) for t in T]
axes[1].plot(T, atm, "o-")
axes[1].set(xlabel="T (years)", ylabel="implied vol (%)", title="ATM-forward term structure")
plt.tight_layout()

## Correlations

In [ ]:
corr = mkt.get_correlation_matrix()
name = f"EQ:{eq.currency.name}:{TICKER}"
if name in corr.names:
    rows = [{"factor": other, "correlation": float(corr.correlation(name, other))} for other in corr.names if other != name]
    display(pd.DataFrame(rows).sort_values("correlation", ascending=False).set_index("factor").round(4))
else:
    print(f"{name} not in the correlation matrix")